# 🇻🇳 PaddleOCR Baseline on Viet-Wiki-Handwriting

**Goal:** simple, open-source baseline. Clone PaddleOCR, run the stock Vietnamese model on `5CD-AI/Viet-Wiki-Handwriting`, measure CER.

**Why PaddleOCR?** Apache-2.0 license, full training code is in the repo — we can customize detector/recognizer/config later.

**Dataset:** 5,796 paragraph-level images (800 px wide), Vietnamese Wikipedia text rendered in handwritten fonts. Gated — requires HF token.

**Runtime:** Colab T4 is enough for inference. Fine-tuning later will want A100.

---

### Secrets expected in Colab (left sidebar → 🔑 Secrets)
- `HF_TOKEN` — HuggingFace read token (you already have access to the gated dataset)
- `GITHUB_TOKEN` — optional, only needed if we push fixes from Colab

## 1. Environment & GPU check

In [ ]:
!nvidia-smi | head -n 20
import sys, platform
print(f"Python: {sys.version.split()[0]}  |  Platform: {platform.platform()}")

## 2. Secrets — HF + GitHub tokens from Colab userdata

In [ ]:
import os

def _get_colab_secret(name, retries=2, sleep=1.0):
    """Fetch a Colab secret; first call can TimeoutException while the grant
    dialog is open. Retry once after the user approves."""
    import time
    from google.colab import userdata
    last = None
    for _ in range(retries + 1):
        try:
            return userdata.get(name)
        except Exception as e:
            last = e
            time.sleep(sleep)
    raise last

try:
    HF_TOKEN = _get_colab_secret('HF_TOKEN')
    try:
        GITHUB_TOKEN = _get_colab_secret('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
except ImportError:
    # local run — fall back to env
    HF_TOKEN = os.environ.get('HF_TOKEN')
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

assert HF_TOKEN, "HF_TOKEN not found — add it to Colab Secrets (🔑 icon) and re-run this cell."
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN  # datasets lib also reads this
print('HF_TOKEN:', 'set ✓' if HF_TOKEN else 'MISSING')
print('GITHUB_TOKEN:', 'set ✓' if GITHUB_TOKEN else 'not set (ok for read-only)')

## 3. Install dependencies

- `paddlepaddle-gpu` — the DL framework PaddleOCR is built on (CUDA 12.x wheels for Colab)
- `paddleocr` — high-level OCR API
- `datasets` — HF dataset loader
- `jiwer` — CER/WER computation
- `python-Levenshtein` — faster edit-distance backend for jiwer

In [ ]:
# Paddle 3.0 + PaddleOCR 3.x (Colab is Python 3.12 — Paddle 2.6 has no 3.12 wheel).
# Paddle 3.0 bundles newer NCCL than Colab's pre-installed torch; force-reinstall torch to rebind.
!pip install -q 'paddlepaddle-gpu==3.0.0' -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ 2>&1 | tail -3
!pip install -q --force-reinstall --no-deps torch 2>&1 | tail -3
!pip install -q 'paddleocr>=3.0' 'datasets>=2.18' jiwer python-Levenshtein 2>&1 | tail -3

import paddle, paddleocr
print(f'Paddle:    {paddle.__version__}  |  CUDA available: {paddle.is_compiled_with_cuda()}')
print(f'PaddleOCR: {paddleocr.__version__}')

## 4. Clone PaddleOCR repo (so we can customize later)

The pip package gives us inference. The **repo** gives us training code, configs, and the recognition model YAMLs we'll eventually fine-tune on Viet-Wiki. We clone into `/content/PaddleOCR`.

In [ ]:
import os, subprocess
PADDLE_DIR = '/content/PaddleOCR'
if not os.path.exists(PADDLE_DIR):
    # Shallow clone — we don't need history
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git {PADDLE_DIR}
else:
    print('PaddleOCR repo already present — skipping clone')

!ls {PADDLE_DIR} | head -n 20
# Configs we'll care about for fine-tuning Vietnamese recognition:
!ls {PADDLE_DIR}/configs/rec/multi_language 2>/dev/null | grep -i -E 'latin|vi' | head -n 10 || true

## 5. Load `5CD-AI/Viet-Wiki-Handwriting`

~972 MB. First run downloads; subsequent runs read from cache.

In [ ]:
from datasets import load_dataset

DATASET_ID = '5CD-AI/Viet-Wiki-Handwriting'
ds = load_dataset(DATASET_ID, split='train', token=HF_TOKEN)
print(f'Rows: {len(ds):,}')
print(f'Columns: {ds.column_names}')
print(f'Sample 0 label: {ds[0]["label"][:120]}...')

## 6. Preview one sample

In [ ]:
import matplotlib.pyplot as plt

sample = ds[0]
img = sample['image']
print(f'Image size: {img.size}  |  mode: {img.mode}')
print(f'Label ({len(sample["label"])} chars):\n{sample["label"]}')

plt.figure(figsize=(14, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Viet-Wiki-Handwriting — sample 0')
plt.show()

## 7. Run stock PaddleOCR (lang=`vi`)

PaddleOCR bundles a multilingual `latin` recognizer that covers Vietnamese via `lang='vi'`. We run the full pipeline (detection → classification → recognition) and concatenate the detected line texts into one paragraph.

> **Known limitation:** the stock `latin` recognizer was not specifically tuned for Vietnamese tone/modifier marks. This is the baseline we want to *beat* by fine-tuning later.

In [ ]:
from paddleocr import PaddleOCR

# PaddleOCR 3.x API. In 3.x: use_angle_cls → use_textline_orientation; use_gpu removed.
ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    lang='vi',
)
print('PaddleOCR initialized (lang=vi)')

## 8. Baseline CER on a small eval subset

Start with `N_EVAL = 20` for a fast smoke-run. Bump to 200+ once the pipeline is clean.

In [ ]:
import numpy as np, random
from tqdm.auto import tqdm
from jiwer import cer, wer

N_EVAL = 20
random.seed(42)
eval_idx = random.sample(range(len(ds)), N_EVAL)

def _extract_texts(res):
    """PaddleOCR 3.x result item — rec_texts either as attr or dict key."""
    if res is None:
        return []
    if hasattr(res, 'get'):
        return res.get('rec_texts') or []
    return getattr(res, 'rec_texts', []) or []

def predict_one(pil_img):
    arr = np.array(pil_img.convert('RGB'))
    result = ocr.predict(input=arr)  # 3.x: predict(), not ocr()
    if not result:
        return ''
    return ' '.join(_extract_texts(result[0]))

preds, refs = [], []
for i in tqdm(eval_idx, desc='PaddleOCR baseline'):
    s = ds[i]
    preds.append(predict_one(s['image']))
    refs.append(s['label'])

baseline_cer = cer(refs, preds)
baseline_wer = wer(refs, preds)
print('\n' + '=' * 60)
print(f'BASELINE — PaddleOCR lang=vi (stock, no fine-tune)')
print('=' * 60)
print(f'N eval:  {N_EVAL}')
print(f'CER:     {baseline_cer:.4f}')
print(f'WER:     {baseline_wer:.4f}')

## 9. Inspect a few predictions

In [ ]:
for k in range(min(3, len(preds))):
    print(f'--- sample {eval_idx[k]} ---')
    print(f'GT   : {refs[k][:200]}')
    print(f'PRED : {preds[k][:200]}')
    print()

## 10. Next steps — where to customize

The cloned `/content/PaddleOCR` repo gives us everything needed to fine-tune:

| What we'd change | Where |
|---|---|
| Detection backbone / DB head | `PaddleOCR/configs/det/` |
| Vietnamese recognition config | `PaddleOCR/configs/rec/multi_language/rec_latin_lite_train.yml` |
| Character dict (add full Vietnamese tone set) | `PaddleOCR/ppocr/utils/dict/` |
| Training entry point | `PaddleOCR/tools/train.py -c <yaml>` |

Next notebook can:
1. Convert Viet-Wiki-Handwriting → PaddleOCR text-recognition format (`label.txt` with `path<TAB>transcript`)
2. Export line-level crops (we'll need to either run detection first, or find a line-level version of the dataset)
3. Fine-tune the Vietnamese recognizer and compare CER vs. this baseline